# 01.4 — Where RAG breaks

Most courses give you a list of things that can go wrong. This one gives you the
measured results of a pipeline that already ran, on the corpus you're about to
use, and asks you to classify the failures yourself.

`results/02-baseline.json` is committed to this repository. It's the output of
module 02 — a deliberately naive pipeline scored against 36 questions. You'll
reproduce it in two modules' time.

## Load the results

In [1]:
import json
from pathlib import Path

results = json.loads(Path('../../results/02-baseline.json').read_text())

print(results['metric'], '=', results['score'])
print(results['pipeline'])

rows = results['results']
print(f"\n{sum(r['hit'] for r in rows)}/{len(rows)} questions retrieved the right document first")

hit@1 = 0.4444
{'documents': 7, 'chunks': 71, 'chunk_size': 500, 'embedding_model': 'BAAI/bge-small-en-v1.5'}

16/36 questions retrieved the right document first


Under half. Now find out why — and resist the assumption you're about to make.

In [2]:
misses = [r for r in rows if not r['hit']]

never_loaded = [r for r in misses if not r['reachable']]
wrong_doc = [r for r in misses if r['reachable']]

print(f'{len(never_loaded)}  the document was never loaded at all')
print(f'{len(wrong_doc)}   a different document ranked first')

print('\nwhen a different document won, which one?\n')
for r in wrong_doc:
    print(f"  {r['id']}  wanted {r['expected']}")
    print(f"       got    {r['returned'][0]}\n")

12  the document was never loaded at all
8   a different document ranked first

when a different document won, which one?

  Q02  wanted sahel-employee-handbook-2025.pdf
       got    sahel-employee-handbook-2023.pdf

  Q03  wanted sahel-employee-handbook-2025.pdf
       got    sahel-employee-handbook-2023.pdf

  Q04  wanted sahel-employee-handbook-2025.pdf
       got    sahel-employee-handbook-2023.pdf

  Q06  wanted sahel-employee-handbook-2025.pdf
       got    sahel-employee-handbook-2023.pdf

  Q07  wanted sahel-employee-handbook-2025.pdf
       got    sahel-employee-handbook-2023.pdf

  Q08  wanted nfsc-circular-2025-02-amendment.pdf
       got    nfsc-circular-2024-07-cybersecurity.pdf

  Q11  wanted nfsc-circular-2025-02-amendment.pdf
       got    nfsc-circular-2024-07-cybersecurity.pdf

  Q14  wanted sahel-procurement-policy-v3.pdf
       got    nfsc-circular-2025-02-amendment.pdf



Look at that second list carefully.

Almost every one returned **an older version of the correct document**. Questions
about the per diem, remote work, probation and notice period return the 2023
employee handbook when the answer is in the 2025 edition. Questions about the
incident reporting window return the 2024 circular when a 2025 amendment changed
it.

One entry is different. Q14 asks who approves a purchase of a particular amount
and returns a regulator's circular — because that circular happens to mention a
similar figure as a reporting threshold. The embedding matched on a number rather
than on meaning.

## Now classify them

In [3]:
# Two questions counted as HITS but cannot actually be answered:
# the figures they ask for exist only inside a chart image.
unanswerable = [r for r in rows if 'chart-only' in r['failure_class']]

ingestion = len(never_loaded) + (len(wrong_doc) - 1) + len(unanswerable)
retrieval = 1
generation = 0

print(f'ingestion  : {ingestion}')
print(f'retrieval  : {retrieval}')
print(f'generation : {generation}   (this scorer never looks at the answer)')
print(f'\ntotal real problems: {ingestion + retrieval + generation}')

ingestion  : 21
retrieval  : 1
generation : 0   (this scorer never looks at the answer)

total real problems: 22


**Twenty-one of twenty-two problems are ingestion failures.**

That is not what people expect. The instinct when RAG gives a bad answer is to
blame retrieval, or the embedding model, or the LLM. On this corpus, one problem
out of twenty-two is a retrieval problem, and none are the model's fault.

Break the ingestion group down and every member is a decision nobody made:

- **12** — the file was never parsed. Spreadsheets, slides, email, a scan.
- **6** — nothing recorded that one document supersedes another.
- **2** — the answer was in an image and the pipeline only reads text.
- **1** — a boundary case, but still upstream of retrieval.

None of these are fixed by a better embedding model. All of them are fixed before
the index is ever built.

This is why module 03 is six notebooks and why it comes before everything else.

## The three stages, and what fails at each

**Ingestion** — the document never arrived, arrived mangled, or arrived without
the context needed to use it correctly.

| Failure | What you see |
| --- | --- |
| Format not supported | Questions about whole documents always fail |
| Parsing corrupts the text | Garbled tokens, missing sections, no error |
| Structure discarded | Table rows with no headers; who-said-what lost |
| Metadata not captured | Can't filter, can't cite, can't delete on request |
| No version tracking | Superseded documents stay retrievable forever |
| Non-text content ignored | Charts, diagrams and scans are invisible |

**Retrieval** — the text is in the index and the search didn't find it.

| Failure | What you see |
| --- | --- |
| Low recall | The right chunk isn't in the results at all |
| Low precision | The right chunk is there, buried under noise |
| Exact terms missed | Product codes, reference numbers, names |
| Ranking | The answer is at position 8 and you asked for 5 |

**Generation** — the right text reached the model and the answer is still wrong.

| Failure | What you see |
| --- | --- |
| Unfaithful | The answer says things the context doesn't |
| No abstention | It answers confidently when the context can't support it |
| Context ignored | The model prefers its training data over your documents |
| Citations wrong | The claim is right, the attribution isn't |

## The diagnostic question

You have a wrong answer. Which stage?

Ask one thing: **is the answer supported by the text that was retrieved?**

**No — the answer says things the context doesn't.** That's a generation failure.
The model improvised. Fix the prompt, the abstention instruction, or the model.

**Yes — the answer faithfully reports the retrieved text, and the text is wrong**
for this question. That's not a generation failure at all. Something upstream
handed the model the wrong paragraph.

The second case has a name worth learning: **faithful incorrectness**. It is what
you just measured. Every one of those superseded-document failures produces a
fluent, correctly-cited, completely wrong answer — with no hallucination
involved. The model did its job. The pipeline handed it a policy that stopped
being true in 2024.

It is the most expensive failure in production RAG, because it is the one that
looks most like success. An obviously hallucinated answer gets caught. A
well-sourced wrong number gets acted on.

## Where each of these gets fixed

| Failure class | Module |
| --- | --- |
| Format not supported, parsing corruption | 03 — Document Ingestion |
| Structure discarded | 03 |
| Metadata, provenance, version tracking | 03 |
| Chunk boundaries destroying meaning | 04 — Chunking |
| Exact terms missed by embeddings | 05, 07 |
| Low recall, ranking | 06 — Evaluation, then 07 — Improving Retrieval |
| Low precision, context assembly | 10 — Context Engineering |
| Unfaithful answers, no abstention | 11 — Grounded Generation |
| Non-text content | 15 — Multimodal |

Keep this table. When something breaks later, the useful question is never
"why is the LLM wrong" — it's which row you're in.

## What's next

You've seen that failures are measurable and that most of them aren't where you'd
guess. Notebook 5 asks the obvious follow-up: how do you know any of that,
without a number to look at?